# 09 - Baseline Model: Logistic Regression

The first model, kept deliberately simple: no tuning. It sets the floor every later model has to beat, and - as it turns out - it is a good detector of where the *data* is difficult.

**Protocol (from notebook 08).** Everything here uses the 12 **development** patients only, scored with the grouped cross-validation folds defined in the split: each beat is predicted by a model that never saw its fold's patients. The 5 test patients are dropped from memory below so they cannot be used by accident, and stay untouched until every model has been built and one has been chosen using these cross-validation results.

## 1. What logistic regression is

For each beat it computes a weighted sum of the features and squashes it into a probability:

`log-odds = b + w1*x1 + w2*x2 + ...`    `P(abnormal) = 1 / (1 + exp(-log-odds))`

- It is trained by finding the weights that minimise the **log-loss** (heavily penalising confident wrong answers), plus an **L2 penalty** that discourages large weights.
- The output is a *probability*, and a **threshold** (0.5 by default) turns it into a decision. The decision boundary is a flat surface (a hyperplane) in feature space: it cannot represent curved or "only when A and B together" rules unless we engineer them into features.
- A weight can be read as "how much the log-odds of abnormal change when this feature goes up by one unit, other features held fixed".

## 2. Why it is a useful baseline

- **Fast and deterministic** - a convex problem with one best answer, so there is no seed to lose and no luck involved.
- **Hard to overfit** with 13 features and 28,000 beats, so its cross-validation score is a trustworthy floor.
- **Interpretable** - the weights show which features it leans on (with a caveat in section 8).
- **A yardstick for complexity.** A forest or boosting model is only justified if it clearly beats this. If it does not, the extra complexity buys nothing.
- **It exposes data problems quickly** - which is what happened here.

## 3. Do the features need scaling?

Yes, for logistic regression, for three reasons:

1. **The L2 penalty is not scale-invariant.** It punishes large weights, and a feature measured in small units (`amp_mean`, spread of 0.035) needs a large weight to matter, so it is penalised far more than one measured in large units (`qrs_fwhm_ms`, spread of 10.7). Scaling puts every feature on an equal footing.
2. **Optimisation is better conditioned** when features have comparable ranges.
3. **Weights become comparable** ("per one standard deviation") instead of depending on arbitrary units.

Tree-based models do not need scaling (they only compare a feature to a threshold), which is why this step belongs to *this* model's pipeline, not to the feature table.

**Heavy tails need handling too.** The RR features reach 11 s and ratios of 16 (the detector-miss gaps found in the EDA), and one such value would distort a standardised feature. So RR features are first clipped to **fixed physiological bounds** (0.3-2.0 s for intervals; a factor of 3 either way for ratios). The bounds come from reasoning, not from the data, so this step learns nothing and cannot leak. The scaler, in contrast, *is* fitted - inside every training fold only.

In [ ]:
import sys
sys.path.append("..")

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_curve

from src.evaluation import flagged_rate_by_symbol, metrics_by_group, out_of_fold_probabilities, summarize
from src.feature_extraction import MODEL_FEATURES, add_record_relative_features, load_beat_table
from src.models import DEFAULT_CLIP_BOUNDS, make_logistic_regression
from src.preprocessing import load_record
from src.splitting import apply_split, load_split

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

table = add_record_relative_features(load_beat_table()).sort_values(["record", "r_peak_sample"]).reset_index(drop=True)
data = apply_split(table, load_split())
dev = data[data.split == "train"].copy()
del table, data                                   # the test rows are gone from this notebook
print(f"development set: {len(dev)} beats, {dev.record.nunique()} patients, {100 * dev.is_abnormal.mean():.1f}% abnormal")
print("model features:", MODEL_FEATURES)

In [ ]:
scales = dev[MODEL_FEATURES].agg(["std", "min", "max"]).T.round(3)
print(f"Spread (std) of the features ranges from {scales['std'].min():.3f} to {scales['std'].max():.3f} - a {scales['std'].max() / scales['std'].min():.0f}x difference")
print("Fixed clipping bounds:", {k: tuple(round(x, 2) for x in v) for k, v in DEFAULT_CLIP_BOUNDS.items()})
scales

## 4. The pipeline

```
clip RR outliers (fixed bounds)  ->  StandardScaler (fit on training folds)  ->  LogisticRegression (L2, C=1.0, default threshold 0.5)
```

Defaults everywhere; the only choice is `max_iter=1000` so the optimiser is never cut short. It is rebuilt and refit inside every cross-validation fold (`src/models.py`, `src/evaluation.py`).

## 5. Training and prediction with grouped cross-validation

`out_of_fold_probabilities` trains on three folds and predicts the fourth, four times, so every development beat gets a probability from a model that never saw its patient. Three versions are run - the baseline, the same without scaling (to *test* the scaling claim rather than only assert it), and one with `class_weight="balanced"` (a single, labelled look at the imbalance question, not a tuning search).

In [ ]:
variants = {
    "scaled (the baseline)": lambda: make_logistic_regression(),
    "unscaled": lambda: make_logistic_regression(scale=False),
    "scaled + class_weight='balanced'": lambda: make_logistic_regression(class_weight="balanced"),
}

oof, warning_counts = {}, {}
for name, factory in variants.items():
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        oof[name] = out_of_fold_probabilities(factory, dev)
    warning_counts[name] = len(caught)

rows = {"always predict Normal (reference)": summarize(dev.is_abnormal, np.zeros(len(dev)))}
rows.update({name: summarize(dev.is_abnormal, p) for name, p in oof.items()})
comparison = pd.DataFrame(rows).T[["accuracy", "precision", "recall", "f1", "false_alarm_rate", "roc_auc", "pr_auc"]].astype(float).round(3)
comparison["convergence warnings"] = pd.Series(warning_counts)
print(f"Prevalence of Abnormal in the development set: {dev.is_abnormal.mean():.3f}  (= the PR-AUC of a model with no skill)")
comparison

Reading the table:

- **Accuracy is nearly useless here.** Never flagging anything scores 0.845; the baseline reaches 0.901 - only 5.6 points more, while missing 42% of the abnormal beats.
- **Scaling matters, moderately.** Without it F1 falls from 0.645 to 0.561, PR-AUC from 0.680 to 0.639, and the false-alarm rate rises from 4.0% to 7.0%. There were no convergence warnings in any variant, so the gap is not an optimiser failure; it is consistent with the L2 penalty treating features in small and large units unequally (the 305-fold spread above). ROC-AUC barely moves (0.762 vs 0.758): the ranking is similar and mostly the operating point changes. So the scaling claim has been tested, not just asserted.
- **`class_weight='balanced'` slides along the trade-off; it is not a free win.** Recall rises from 0.58 to 0.65, but precision falls from 0.73 to 0.50 and false alarms roughly triple (4.0% to 12.2%), so F1 is lower (0.565). Its ROC-AUC (0.795) and PR-AUC (0.709) are higher, which means the fitted weights themselves changed, not just the cut-off. The default stays the baseline; class weighting and the threshold are choices to make deliberately later, with this trade-off in view.

## 6. Evaluation

Metrics at the default 0.5 threshold, with "abnormal" as the positive class: **precision** (of the beats flagged abnormal, how many were), **recall** (of the abnormal beats, how many were flagged), **F1** (their harmonic mean), and the **false-alarm rate** (share of Normal beats wrongly flagged). ROC-AUC and PR-AUC do not depend on the threshold. Accuracy is shown only to be distrusted: the reference row shows that a model that never flags anything already reaches 84.5%.

In [ ]:
baseline = oof["scaled (the baseline)"]
m = summarize(dev.is_abnormal, baseline)
cm = confusion_matrix(dev.is_abnormal, (baseline >= 0.5).astype(int), labels=[0, 1])

print(f"Baseline, pooled over all out-of-fold predictions (threshold 0.5)")
print(f"  accuracy {m['accuracy']:.3f} | precision {m['precision']:.3f} | recall {m['recall']:.3f} | F1 {m['f1']:.3f}")
print(f"  false-alarm rate {m['false_alarm_rate']:.3f} | ROC-AUC {m['roc_auc']:.3f} | PR-AUC {m['pr_auc']:.3f}")
print(f"  confusion matrix: {m['tn']:,} true Normal, {m['fp']:,} false alarms, {m['fn']:,} missed abnormal, {m['tp']:,} caught abnormal")

In [ ]:
symbols = flagged_rate_by_symbol(dev, baseline)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

ax = axes[0]
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}\n({100 * cm[i, j] / cm[i].sum():.0f}% of the row)", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1], ["predicted Normal", "predicted Abnormal"])
ax.set_yticks([0, 1], ["true Normal", "true Abnormal"])
ax.set_title("Confusion matrix (out-of-fold, threshold 0.5)")

ax = axes[1]
precision, recall, _ = precision_recall_curve(dev.is_abnormal, baseline)
ax.plot(recall, precision, color="tab:blue", label="logistic regression")
ax.axhline(dev.is_abnormal.mean(), color="grey", linestyle="--", label=f"no skill ({dev.is_abnormal.mean():.3f})")
ax.scatter(m["recall"], m["precision"], color="red", zorder=3, label="threshold 0.5")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-recall curve (PR-AUC {m['pr_auc']:.2f})")
ax.legend(loc="upper right")

ax = axes[2]
plot_symbols = symbols.sort_values("pct_flagged_abnormal")
ax.barh(plot_symbols.index, plot_symbols.pct_flagged_abnormal,
        color=["tab:red" if l == "Abnormal" else "tab:blue" for l in plot_symbols.label])
for i, (sym, row) in enumerate(plot_symbols.iterrows()):
    ax.text(row.pct_flagged_abnormal + 1, i, f"n={int(row.beats):,}", va="center", fontsize=8)
ax.set_xlim(0, 118)
ax.set_xlabel("% of beats the model flags as abnormal")
ax.set_title("By beat type (red = truly Abnormal: this is recall;\nblue = truly Normal: this is the false-alarm rate)", fontsize=10)

fig.tight_layout()
fig.savefig("../results/figures/19_logreg_evaluation.png", dpi=120)
plt.show()

**Overall (pooled, out-of-fold):** precision 0.73, recall 0.58, F1 0.65, false-alarm rate 4.0%, ROC-AUC 0.76, PR-AUC 0.68 against a no-skill 0.155. That is real skill - far above chance - but at the default threshold the model misses 42% of the abnormal beats (1,828). A reasonable but unimpressive floor.

**The right-hand panel is the most informative.** The model flags 98.9% of `V` (ventricular) beats and all 16 `a` beats, but only 19.8% of `F`, 10.4% of `A` and 1.2% of `J`. On the Normal side the false-alarm rate is 4.8% for `N` and about 0 for the bundle-branch-block beats (`L` 0.1%, `R` 0%). So "abnormal" is really two different problems for this model: *ventricular* beats (wide, early, followed by a pause - easy) and *atrial/junctional* beats (narrow like normal beats, so only their timing can give them away - mostly missed). The long flat tail of the precision-recall curve is the same fact: from recall of about 0.75 onward, precision falls to the no-skill line, meaning roughly a quarter of the abnormal beats cannot be ranked above normal ones by a linear model on these features.

### The pooled number hides large differences

Per cross-validation fold (each fold is three patients) and per patient. Recall the fold structure from notebook 08: fold 1 holds record 232, the source of most atrial beats, so when it is the validation fold the model trains with almost no atrial examples.

In [ ]:
columns = ["beats", "abnormal", "precision", "recall", "f1", "false_alarm_rate", "roc_auc"]
print("Per cross-validation fold")
display_fold = metrics_by_group(dev, baseline, "cv_fold")[columns].astype(float).round(3)
display_fold

In [ ]:
print("Per patient")
metrics_by_group(dev, baseline, "record")[columns].astype(float).round(3)

In [ ]:
atrial = dev[dev.symbol == "A"].assign(flagged=(baseline >= 0.5).astype(int))
atrial_by_patient = atrial.groupby(["cv_fold", "record"]).agg(atrial_beats=("flagged", "size"), flagged_as_abnormal=("flagged", "sum"))
atrial_by_patient["pct_flagged"] = (100 * atrial_by_patient.flagged_as_abnormal / atrial_by_patient.atrial_beats).round(1)
print("Atrial (A) beats: how many does the model flag as abnormal? (patients with at least 5 atrial beats)")
atrial_by_patient[atrial_by_patient.atrial_beats >= 5]

The pooled F1 of 0.65 averages over very different situations:

- **By fold**, F1 runs from 0.21 to 0.90. Fold 1 (record 232's atrial beats) has precision 0.96 but recall 0.12; fold 2 has recall 0.99 but a 14% false-alarm rate (records 200 and 202); fold 3 is the best (F1 0.90).
- **By patient**, recall is 0.98-0.99 for records 200, 214 and 233 (ventricular beats dominate) but 0.07 for 232, 0.04 for 220 and 0.02 for 234 (atrial and junctional beats). Record 202 has recall 0.94 but precision 0.075: 27.5% of its Normal beats are flagged.
- **Atrial beats are the weak spot, and it is not just the fold structure.** Record 220's 94 atrial beats are scored by a model whose training data *does* include record 232's 1,362 atrial beats - yet only 4 of them are flagged. The two records where most atrial beats are "caught" (200: 24 of 30; 202: 33 of 35) are the two with the highest false-alarm rates (15% and 27.5%), so those catches look like the model flagging anything in irregular stretches rather than recognising atrial beats.

A single pooled number would hide all of this, so **every later model should be reported the same way: per fold and per patient.**

### Where do the false alarms come from?

The annotations also carry *rhythm* labels (`(N` sinus rhythm, `(AFIB` atrial fibrillation, `(AFL` atrial flutter, `(B` bigeminy, ...). Assign each Normal beat the rhythm in force when it occurred and see where the model raises false alarms. This uses annotations for **analysis only** - they are not model inputs.

In [ ]:
rhythm = pd.Series("(unlabeled", index=dev.index)
for rec, part in dev.groupby("record"):
    _, ann = load_record(rec)
    symbols_, samples_, notes_ = np.asarray(ann.symbol), np.asarray(ann.sample), np.asarray(ann.aux_note)
    is_change = symbols_ == "+"
    starts, names = samples_[is_change], [n.strip("\x00").strip() for n in notes_[is_change]]
    position = np.searchsorted(starts, part.r_peak_sample.to_numpy(), side="right") - 1
    rhythm.loc[part.index] = [names[i] if i >= 0 else "(unlabeled" for i in position]

normal_beats = dev[dev.label == "Normal"].assign(rhythm=rhythm, false_alarm=(baseline >= 0.5).astype(int))
by_rhythm = normal_beats.groupby("rhythm").agg(normal_beats=("false_alarm", "size"), false_alarms=("false_alarm", "sum"))
by_rhythm["false_alarm_pct"] = (100 * by_rhythm.false_alarms / by_rhythm.normal_beats).round(1)
by_rhythm["share_of_all_false_alarms_pct"] = (100 * by_rhythm.false_alarms / by_rhythm.false_alarms.sum()).round(1)
by_rhythm.sort_values("normal_beats", ascending=False)

Most false alarms are not random: **in sinus rhythm (`(N`) the false-alarm rate is 1.9%, but in atrial fibrillation it is 17.4% and in atrial flutter 93.3%.** Those two rhythms hold 12% of the Normal beats (2,781 of 23,633) yet produce **59% of all false alarms**. Bigeminy and the other rhythms are almost never flagged.

The cause is the labeling scheme meeting a timing-based model. In atrial fibrillation each individual beat has a normal-looking QRS, so its beat-level label is Normal - but its timing is irregular, and irregular timing is exactly what the model learned to associate with premature beats. A model that sees a beat and its two neighbouring intervals cannot tell "one premature beat in a regular rhythm" from "every beat is irregular". This is a real limitation of both the features and the beat-level framing, to state in the README. A natural improvement is rhythm-context features (for example the variability of the last several RR intervals), but that is a design change to make deliberately, on development-set evidence like this - not something to bolt on now.

## 7. How the weights look (a model fitted on all development beats, for inspection only)

In [ ]:
final_model = make_logistic_regression().fit(dev[MODEL_FEATURES], dev.is_abnormal)
coefficients = pd.Series(final_model[-1].coef_[0], index=MODEL_FEATURES).sort_values()
weights = pd.DataFrame({"weight_per_1_SD": coefficients.round(3), "odds_ratio_per_1_SD": np.exp(coefficients).round(3)})
print("intercept (log-odds of abnormal at the average beat):", final_model[-1].intercept_.round(3))

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.barh(coefficients.index, coefficients.values, color=["tab:red" if c > 0 else "tab:blue" for c in coefficients])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Weight per +1 standard deviation (positive = pushes toward Abnormal)")
ax.set_title("Logistic regression weights - do not read them one by one (see text)")
fig.tight_layout()
fig.savefig("../results/figures/20_logreg_weights.png", dpi=120)
plt.show()
weights.T

Do not read these one by one. The amplitude features are strongly correlated with each other (notebook 07: `amp_std`, `amp_max` and `qrs_p2p_mv` correlate 0.76-0.87), and with correlated features logistic regression can split one effect across them with large weights of opposite sign: `amp_max` -9.8 against `amp_std` +6.3 and `qrs_p2p_mv` +4.2, which largely cancel. The resulting "odds ratios" (from 0.00 to 539 per standard deviation) are artefacts, not findings. Likewise `qrs_fwhm_ms` gets a *negative* weight although wider beats are more often abnormal on their own (single-feature AUC 0.76 in the EDA): once the correlated amplitude features are in the model, the sign of what is left over says nothing about the feature by itself.

What is safe to take away: the timing weights that are not swamped by collinearity (`rr_ratio` and `rr_pre_rel`, both negative: shorter-than-usual intervals push toward abnormal) match the physiology, and the model as a whole works even though its individual coefficients are unreadable. The practical remedies are fewer, less redundant features, or a model that does not depend on coefficients (trees) - questions for the model comparison, not for tuning this one.

## Summary

**Model:** clip RR outliers (fixed bounds) -> standardise -> logistic regression (L2, C=1, threshold 0.5), untuned. Evaluated with grouped cross-validation on the 12 development patients; the test set is untouched.

**Result (pooled, out-of-fold):** accuracy 0.90, precision 0.73, recall 0.58, F1 0.65, false-alarm rate 4.0%, ROC-AUC 0.76, PR-AUC 0.68 (no-skill 0.155). A real but modest floor.

**What the baseline told us about the data**

1. Ventricular beats are found (98.9%); atrial and junctional beats mostly are not (10% and 1%): they look like normal beats except for their timing, and the timing features alone do not carry across patients.
2. 59% of the false alarms come from atrial fibrillation/flutter stretches, where every interval is irregular but the beats are labeled Normal.
3. Results vary enormously by fold (F1 0.21-0.90) and by patient - report per fold and per patient from now on.
4. Scaling helped (F1 0.56 -> 0.65); class weighting trades precision for recall and is a later decision.

**What the next models have to beat, and why they might:** a decision tree or forest can use interactions (early *and* wide) and is unaffected by scaling and collinearity; boosting can go further. The question is not only whether the pooled F1 rises, but whether they help with the two real problems - atrial beats and irregular rhythm.